In [2]:
!pip install akshare
!pip install yfinance # Install yfinance as well
import akshare as ak
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
import zipfile
import time
import yfinance as yf # Import yfinance

stock_codes_map = {
    "300921": {"akshare": "300921", "yfinance": "300921.SZ"},
    "002285": {"akshare": "002285", "yfinance": "002285.SZ"},
    # "301667": {"akshare": "301667", "yfinance": "301667.SZ"},
    "688331": {"akshare": "688331", "yfinance": "688331.SS"},
    "603538": {"akshare": "603538", "yfinance": "603538.SS"},
    "600726": {"akshare": "600726", "yfinance": "600726.SS"},
    # "300685": {"akshare": "300685", "yfinance": "300685.SZ"},
    # "301389": {"akshare": "301389", "yfinance": "301389.SZ"},
    "300724": {"akshare": "300724", "yfinance": "300724.SZ"},
    # "300643": {"akshare": "300643", "yfinance": "300643.SZ"},
    # "301158": {"akshare": "301158", "yfinance": "301158.SZ"},
    # "sh000001": {"akshare": "sh000001", "yfinance": "000001.SS"} # Add yfinance symbol for index
}

stock_codes = list(stock_codes_map.keys())

print(f"🔥 已加载股票代码列表，共 {len(stock_codes)} 支股票")

# 时间范围
start_date = "20210601"
end_date = datetime.today().strftime("%Y%m%d")

# 保存路径
save_path = "./data/"
os.makedirs(save_path, exist_ok=True)

# 字段标准化函数：中文 → 英文
def standardize_columns(df):
    column_mapping = {
        "日期": "Date", "股票代码": "Code", "开盘": "Open", "收盘": "Close",
        "最高": "High", "最低": "Low", "成交量": "Volume", "成交额": "Amount",
        "振幅": "Amplitude", "涨跌幅": "ChangePct", "涨跌额": "ChangeAmt", "换手率": "Turnover"
    }
    # 只重命名df中存在的列
    df.rename(columns={col_cn: col_en for col_cn, col_en in column_mapping.items() if col_cn in df.columns}, inplace=True)
    return df

# RSI计算函数
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# ATR计算函数
def compute_atr(df, window=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.ewm(span=window, adjust=False).mean()
    return atr

# OBV计算函数
def compute_obv(df):
    obv = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    return obv


# 数据加载与特征工程
def load_and_prepare_data(file_path):
    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df.set_index("Date", inplace=True)

    # 针对yfinance数据中可能存在的全NaN列进行填充，以防止dropna清空整个DataFrame
    for col in ['Amount', 'Amplitude', 'ChangePct', 'ChangeAmt', 'Turnover']:
        if col in df.columns and df[col].isnull().all():
            df[col] = 0 # 填充0，因为这些对于指数数据通常不提供或无意义

    # 添加衍生特征
    if "Close" in df.columns:
        df["MA5"] = df["Close"].rolling(window=5).mean()
        df["MA10"] = df["Close"].rolling(window=10).mean()
        df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
        df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()
        df["MACD"] = df["EMA12"] - df["EMA26"]
        df["RSI"] = compute_rsi(df["Close"], window=14)
        df["Return_1d"] = df["Close"].pct_change()
        df["Volatility_5d"] = df["Close"].rolling(window=5).std()
        df['Close_Lag1'] = df['Close'].shift(1)

    if "High" in df.columns and "Low" in df.columns and "Close" in df.columns:
        df["ATR"] = compute_atr(df, window=14) # 添加ATR

    if "Close" in df.columns and "Volume" in df.columns:
        df["OBV"] = compute_obv(df) # 添加OBV
        df['Volume_Lag1'] = df['Volume'].shift(1)

    df["Weekday"] = df.index.weekday
    df["IsMonthEnd"] = df.index.is_month_end.astype(int)

    df.dropna(inplace=True)

    # 定义一个全面的潜在特征列表
    potential_features = [
        "Open", "High", "Low", "Close", "Volume", "Amount", "Turnover",
        "MA5", "MA10", "EMA12", "EMA26", "MACD", "RSI",
        "Return_1d", "Volatility_5d", "Weekday", "IsMonthEnd",
        "ATR", "OBV", "Close_Lag1", "Volume_Lag1"
    ]

    # 根据DataFrame中实际存在的列来筛选特征
    features = [f for f in potential_features if f in df.columns]

    if not features or "Close" not in features:
        raise ValueError(f"Insufficient features or missing 'Close' column for {os.path.basename(file_path)} after processing.")

    print(f"\nProcessed data for {os.path.basename(file_path)} (last 5 rows):")
    print(df[features].tail())
    print("-" * 30)


    scaler = MinMaxScaler()
    df_scaled = scaler.fit_transform(df[features])

    return df_scaled, df["Close"].values, scaler, features # 返回features列表

# 构造时间序列样本
def create_sequences(data_scaled, close_prices, window_size=60, pred_size=10): # 修改pred_size为10
    X, y = [], []
    for i in range(len(data_scaled) - window_size - pred_size + 1):
        X.append(data_scaled[i:i+window_size])
        y.append(close_prices[i+window_size:i+window_size+pred_size])
    return np.array(X), np.array(y)

# 构建LSTM模型
def build_lstm_model(input_shape, output_size=10): # 修改output_size为10
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2)) # 添加 Dropout 层
    model.add(LSTM(32))
    model.add(Dropout(0.2)) # 添加 Dropout 层
    model.add(Dense(output_size))
    model.compile(optimizer='adam', loss='mse')
    return model

# 模型训练
def train_lstm_model(X, y, input_shape, model_path):
    model = build_lstm_model(input_shape, output_size=y.shape[1]) # 根据实际输出维度设置output_size
    checkpoint = ModelCheckpoint(
    model_path.replace(".h5", ".keras"),  # 自动替换扩展名
    monitor='loss', # 监控训练损失
    save_best_only=True, # 只保存最优模型
    verbose=1
)
    early_stopping = EarlyStopping(
        monitor='loss', # 监控训练损失
        patience=15,    # 容忍没有进步的训练轮次
        verbose=1,      # 显示早停信息
        restore_best_weights=True # 恢复最佳模型权重
    )


    model.fit(X, y, epochs=150, batch_size=32, callbacks=[checkpoint, early_stopping], verbose=0)
    return model

# 预测并绘图
def predict_and_plot(model, scaled_data, scaler, window_size, stock_code, features, output_dir="./charts"): # 接收features列表
    last_window = scaled_data[-window_size:]
    prediction = model.predict(np.expand_dims(last_window, axis=0))[0]

    # 恢复预测价格时，需要构建一个包含所有特征的dummy数组
    # 找到Close列在features列表中的索引
    close_idx = features.index("Close")
    num_features = len(features)

    full_dummy = np.zeros((10, num_features)) # 修改为预测10天

    # 将最后一个时间窗口的特征值复制到dummy数组中
    # 注意：这里假设最后一个时间窗口的特征值可以用于未来预测，这可能需要更复杂的处理
    # 更准确的做法是根据预测的收盘价，重新计算未来日期的一些特征（如MA, EMA等)，但这超出了当前修改范围
    # 这里简化处理，仅用最后一个已知时间点的信息进行逆缩放
    reference = scaled_data[-1]
    for i in range(10): # 修改为预测10天
        full_dummy[i, :] = reference
        full_dummy[i, close_idx] = prediction[i]


    recovered = scaler.inverse_transform(full_dummy)[:, close_idx] # 根据Close列的索引进行逆缩放

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    chart_path = os.path.join(output_dir, f"{stock_code}_{datetime.today().strftime('%Y-%m-%d')}_forecast.png")


    plt.figure(figsize=(10, 4))
    plt.plot(range(1, 11), recovered, marker='o', label="预测收盘价") # 修改x轴范围
    plt.title(f"{stock_code} - 未来10天收盘价预测") # 修改标题
    plt.xlabel("未来天数")
    plt.ylabel("收盘价")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(chart_path)
    plt.close()

    print(f"✅ 图表已保存：{chart_path}")
    return recovered

# 打包所有图表为 zip 文件：
def zip_today_charts(chart_dir="./charts", zip_name="charts_output.zip"):
    today_str = datetime.today().strftime("%Y-%m-%d")
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        for file in os.listdir(chart_dir):
            if file.endswith(".png") and today_str in file:
                zipf.write(os.path.join(chart_dir, file), arcname=file)
    print(f"📦 今日图表已打包为：{zip_name}")


# 🔄 抓取历史数据并保存为 CSV
for code in stock_codes:
    akshare_symbol = stock_codes_map[code].get("akshare")
    yfinance_symbol = stock_codes_map[code].get("yfinance")
    print(f"📈 正在处理：{code}")
    df = None

    # Try fetching with AkShare first
    try:
        print(f"尝试使用 AkShare 下载 {code} ({akshare_symbol})...")
        if akshare_symbol.startswith("sh") or akshare_symbol.startswith("sz"): # 判断是否为指数代码
            df = ak.stock_zh_index_daily(symbol=akshare_symbol[-6:]) # Use sliced symbol for akshare
            if df is not None and not df.empty:
                df = df[(df['日期'] >= start_date) & (df['日期'] <= end_date)] # 过滤日期
        else:
            df = ak.stock_zh_a_hist(symbol=akshare_symbol, period="daily", start_date=start_date, end_date=end_date, adjust="qfq")

        if df is not None and not df.empty:
            df["股票代码"] = code
            df = standardize_columns(df)
            print(f"✅ {code} (AkShare) 获取成功，共 {len(df)} 条记录")
        else:
            print(f"⚠️ {code} (AkShare) 无数据或数据为空，尝试使用 yfinance。")
            df = None # Reset df to None to trigger yfinance fallback
    except Exception as e:
        print(f"❌ {code} (AkShare) 抓取失败：{e}，尝试使用 yfinance。")
        df = None # Reset df to None to trigger yfinance fallback

    # If AkShare failed or returned empty, try fetching with yfinance as a fallback
    if (df is None or df.empty) and yfinance_symbol: # Check if df is still None or empty
        try:
            print(f"🔄 尝试使用 yfinance 下载 {code} ({yfinance_symbol})...")
            # yfinance expects YYYY-MM-DD format
            yf_start_date = start_date[:4] + '-' + start_date[4:6] + '-' + start_date[6:]
            yf_end_date = end_date[:4] + '-' + end_date[4:6] + '-' + end_date[6:]

            df_yf = yf.download(yfinance_symbol, start=yf_start_date, end=yf_end_date, progress=False) # progress=False to reduce verbose output

            if not df_yf.empty:
                df_yf.reset_index(inplace=True) # Date 是 yfinance 中的索引，转换为列

                df_processed_yf = pd.DataFrame()
                df_processed_yf['Date'] = df_yf['Date']
                df_processed_yf['Code'] = code
                df_processed_yf['Open'] = df_yf['Open']
                df_processed_yf['High'] = df_yf['High']
                df_processed_yf['Low'] = df_yf['Low']
                df_processed_yf['Close'] = df_yf['Close'] # Use Close, 而非 Adj Close
                df_processed_yf['Volume'] = df_yf['Volume']

                # Fill NaN columns with 0 or NaN as they might not be available from yfinance
                df_processed_yf['Amount'] = np.nan
                df_processed_yf['Amplitude'] = np.nan
                df_processed_yf['ChangePct'] = np.nan
                df_processed_yf['ChangeAmt'] = np.nan
                df_processed_yf['Turnover'] = np.nan

                df = df_processed_yf
                df = standardize_columns(df) # Standardize columns after adding 'Code'
                print(f"✅ {code} (yfinance) 获取成功，共 {len(df)} 条记录")
            else:
                print(f"⚠️ {code} (yfinance) 未获取到数据。")
                df = None
        except Exception as e:
            print(f"❌ {code} (yfinance) 抓取失败：{e}")
            df = None

    if df is not None and not df.empty:
        df.to_csv(f"{save_path}{code}.csv", index=False)
    else:
        print(f"❌ {code} 数据获取失败，跳过保存。")
    time.sleep(5) # Add a 5-second delay between requests

# 🔄 加载并标准化所有股票数据
scaled_stock_data = {}
for code in stock_codes:
    file_path = f"{save_path}{code}.csv"
    if os.path.exists(file_path):
        try:
            scaled_data, close_prices, scaler, features = load_and_prepare_data(file_path) # 接收features列表
            # 在这里不需要重新构建df_scaled，直接使用scaled_data即可
            scaled_stock_data[code] = (scaled_data, close_prices, scaler, features) # 存储features列表
        except Exception as e:
            print(f"❌ 数据处理失败：{code} → {e}")
    else:
        print(f"⚠️ 缺失数据文件：{file_path}")

# 🔁 构造训练样本并训练模型
os.makedirs("./models", exist_ok=True)
chart_dir = "./charts"

for code in stock_codes:
    if code not in scaled_stock_data:
        continue

    scaled_data, close_prices, scaler, features = scaled_stock_data[code] # 获取features列表

    X, y = create_sequences(scaled_data, close_prices, window_size=60, pred_size=10) # 修改pred_size为10
    if X.shape[0] == 0:
        print(f"⚠️ 样本数量不足，跳过训练：{code}")
        continue

    model_path = f"./models/{code}_lstm.h5"
    model = train_lstm_model(X, y, input_shape=X.shape[1:], model_path=model_path)
    predict_and_plot(model, scaled_data, scaler, window_size=60, stock_code=code, features=features, output_dir=chart_dir) # 传递features列表

zip_today_charts(chart_dir)
# 俩部分代码分开执行

🔥 已加载股票代码列表，共 6 支股票
📈 正在处理：300921
尝试使用 AkShare 下载 300921 (300921)...
✅ 300921 (AkShare) 获取成功，共 1172 条记录
📈 正在处理：002285
尝试使用 AkShare 下载 002285 (002285)...
❌ 002285 (AkShare) 抓取失败：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))，尝试使用 yfinance。
🔄 尝试使用 yfinance 下载 002285 (002285.SZ)...
✅ 002285 (yfinance) 获取成功，共 1171 条记录


/tmp/ipykernel_4239/534462443.py:267: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_yf = yf.download(yfinance_symbol, start=yf_start_date, end=yf_end_date, progress=False) # progress=False to reduce verbose output


📈 正在处理：688331
尝试使用 AkShare 下载 688331 (688331)...
❌ 688331 (AkShare) 抓取失败：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))，尝试使用 yfinance。
🔄 尝试使用 yfinance 下载 688331 (688331.SS)...
✅ 688331 (yfinance) 获取成功，共 968 条记录


/tmp/ipykernel_4239/534462443.py:267: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_yf = yf.download(yfinance_symbol, start=yf_start_date, end=yf_end_date, progress=False) # progress=False to reduce verbose output


📈 正在处理：603538
尝试使用 AkShare 下载 603538 (603538)...
❌ 603538 (AkShare) 抓取失败：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))，尝试使用 yfinance。
🔄 尝试使用 yfinance 下载 603538 (603538.SS)...
✅ 603538 (yfinance) 获取成功，共 1171 条记录


...

/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26469 (\N{CJK UNIFIED IDEOGRAPH-6765}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 22825 (\N{CJK UNIFIED IDEOGRAPH-5929}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25910 (\N{CJK UNIFIED IDEOGRAPH-6536}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 30424 (\N{CJK UNIFIED IDEOGRAPH-76D8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 20215 (\N{CJK UNIFIED IDEOGRAPH-4EF7}

✅ 图表已保存：./charts/603538_2026-04-01_forecast.png

Epoch 1: loss improved from None to 2.95954, saving model to ./models/600726_lstm.keras

Epoch 1: finished saving model to ./models/600726_lstm.keras

Epoch 2: loss improved from 2.95954 to 0.50575, saving model to ./models/600726_lstm.keras

Epoch 2: finished saving model to ./models/600726_lstm.keras

Epoch 3: loss improved from 0.50575 to 0.43235, saving model to ./models/600726_lstm.keras

Epoch 3: finished saving model to ./models/600726_lstm.keras

Epoch 4: loss improved from 0.43235 to 0.40293, saving model to ./models/600726_lstm.keras

Epoch 4: finished saving model to ./models/600726_lstm.keras

Epoch 5: loss improved from 0.40293 to 0.36778, saving model to ./models/600726_lstm.keras

Epoch 5: finished saving model to ./models/600726_lstm.keras

Epoch 6: loss improved from 0.36778 to 0.33504, saving model to ./models/600726_lstm.keras

Epoch 6: finished saving model to ./models/600726_lstm.keras

Epoch 7: loss improved from 0.

/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26469 (\N{CJK UNIFIED IDEOGRAPH-6765}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 22825 (\N{CJK UNIFIED IDEOGRAPH-5929}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25910 (\N{CJK UNIFIED IDEOGRAPH-6536}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 30424 (\N{CJK UNIFIED IDEOGRAPH-76D8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 20215 (\N{CJK UNIFIED IDEOGRAPH-4EF7}

✅ 图表已保存：./charts/600726_2026-04-01_forecast.png

Epoch 1: loss improved from None to 7778.17822, saving model to ./models/300724_lstm.keras

Epoch 1: finished saving model to ./models/300724_lstm.keras

Epoch 2: loss improved from 7778.17822 to 7439.76562, saving model to ./models/300724_lstm.keras

Epoch 2: finished saving model to ./models/300724_lstm.keras

Epoch 3: loss improved from 7439.76562 to 7192.74463, saving model to ./models/300724_lstm.keras

Epoch 3: finished saving model to ./models/300724_lstm.keras

Epoch 4: loss improved from 7192.74463 to 6974.15723, saving model to ./models/300724_lstm.keras

Epoch 4: finished saving model to ./models/300724_lstm.keras

Epoch 5: loss improved from 6974.15723 to 6777.81641, saving model to ./models/300724_lstm.keras

Epoch 5: finished saving model to ./models/300724_lstm.keras

Epoch 6: loss improved from 6777.81641 to 6592.14404, saving model to ./models/300724_lstm.keras

Epoch 6: finished saving model to ./models/300724_lstm.kera

/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26410 (\N{CJK UNIFIED IDEOGRAPH-672A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 26469 (\N{CJK UNIFIED IDEOGRAPH-6765}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 22825 (\N{CJK UNIFIED IDEOGRAPH-5929}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 25910 (\N{CJK UNIFIED IDEOGRAPH-6536}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 30424 (\N{CJK UNIFIED IDEOGRAPH-76D8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_4239/534462443.py:214: UserWarning: Glyph 20215 (\N{CJK UNIFIED IDEOGRAPH-4EF7}